In [192]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import sys
import json
from datetime import datetime

In [193]:
from typing import get_origin, get_args, Literal

In [194]:
load_dotenv()

OPEN_ROUTER_API = os.getenv("OPENROUTER_API_KEY") or os.getenv("OPEN_ROUTER_API_KEY")
MODEL = os.getenv("OPENROUTER_MODEL") or os.getenv("MODEL") or "google/gemini-2.5-flash"

if not OPEN_ROUTER_API:
    raise ValueError("API key not found. Please set OPENROUTER_API_KEY in .env file.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPEN_ROUTER_API,
)


In [195]:
class Memory:
    def __init__(self):
        self.data = {}

    def remember(self, key, value):
        self.data[key] = value
    
    def recall(self, key):
        return self.data.get(key)

    def forget(self, key):
        if key in self.data:
            del self.data[key]

In [196]:
class Tool:
    def __init__(self, function, description):
        self.function = function
        self.description = description
        self.parameters = generate_parameters(function)

    def execute(self, arguments, context):
        try:
            if context is None:
                context = {}
            
            return self.function(**arguments, **context)
        except Exception as e:
            return f"Tool execution failed: {str(e)}"

    def schema(self):
        return {
            "type" : "function",
            "function":{
                "name": self.function.__name__,
                "description": self.description,
                "parameters": self.parameters
            }
        }
    

In [197]:
class Agent:
    def __init__(self, client, tool_list, model, system_prompt):
        self.client = client
        self.tool_list = tool_list
        self.model = model

        self.TOOL_MAP = {
            tool.function.__name__ : tool
            for tool in tool_list
        }

        self.Tool_SCHEMA = [
            tool.schema()
            for tool in tool_list
        ]

        self.messages = [{
            "role": "system",
            "content": system_prompt
        }       
        ]

        self.state = {
        }

        self.memory = Memory()
        
        self.context = {
            "memory" : self.memory
        }
        
    def set_state(self, key, value):
        self.state[key] = value

    def get_state(self, key):
        return self.state[key]

    def call_llm(self):
        return client.chat.completions.create(
                model=self.model,
                messages=self.messages,
                tools=self.Tool_SCHEMA,
                max_tokens=1000
            )

    def execute(self, tool_call):
            tool_name = tool_call.function.name

            tool = self.TOOL_MAP.get(tool_name)

            if not tool:
                return f"Tool '{tool_name}' does not exist."

            try:
                arguments = json.loads(tool_call.function.arguments)
                return tool.execute(arguments, self.context)

            except Exception as e:
                return f"Tool execution failed: {str(e)}"
        
    
    def run(self, user_input):
        self.messages.append({"role": "user", "content": user_input})

        MAX_ITERATIONS = 10

        for iteration in range(MAX_ITERATIONS):
            
            response = self.call_llm()

            response_message = response.choices[0].message
            self.messages.append(response_message)

            if not response_message.tool_calls:
                return f"AI: ",response_message.content

            for tool_call in response_message.tool_calls:
                    print("Tool is running")
                    result = self.execute(tool_call)
                    
                    self.messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": tool_call.function.name,
                        "content": json.dumps(result)
                    })

        
        else:
            print("MAX Tool iterations reached!!")
            
        

In [198]:
def python_type_to_json_type(annotation):

    if get_origin(annotation) is Literal:

        values = get_args(annotation)

        first_value = values[0]

        if isinstance(first_value, str):
            json_type = "string"
        
        elif isinstance(first_value, int):
            json_type = "integer"

        elif isinstance(first_value, float):
            json_type = "number"
        
        elif isinstance(first_value, bool):
            json_type = "boolean"

        else:
            json_type = "string"

        return{
            "type": json_type,
            "enum": list(values)
        }
    if annotation == str:
        return "string"

    elif annotation == int:
        return "integer"

    elif annotation == float:
        return "number"

    elif annotation == bool:
        return "boolean"

    return "string"

In [199]:
import inspect

def generate_parameters(function):
    
    signature = inspect.signature(function)

    properties = {}
    required = []

    for name, parameter in signature.parameters.items():
        if name == "memory":
            continue

        json_type = python_type_to_json_type(parameter.annotation)

        properties[name] = {
            "type" : json_type
        }

        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    return {
        "type" : "object",
        "properties" : properties,
        "required" : required
    }

In [200]:
def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [201]:
time_tool = Tool(
    function = get_current_time,
    description="Get current Time",
)

In [202]:
def calculator(a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]):
    if(operation == "add"):
        return a+b

    elif(operation == "divide"):
        if(b != 0):
            return a/b
        else: return "Cannot divide with Zero"

    elif(operation == "subtract"):
        return a-b
    
    elif(operation == "multiply"):
        return a*b


In [203]:
calculator_tool = Tool(
    function=calculator,
    description="Perform mathematical calculations",
)

In [204]:
def greet(name: str, age: int, excited: bool = False):
    if excited:
        return f"Hello {name}! You are {age} years old!"
    return f"Hello {name}. You are {age} years old."

In [205]:
greet_tool = Tool(
    greet,
    "Greets a Person"
)

In [206]:
def save_memory(memory: Memory, key: str, value: str):
    memory.remember(key, value)
    return f"Remembered {key} = {value}"

In [207]:
memory_tool = Tool(
    save_memory,
    "Saves important information to agent's memory"
)

In [208]:
def recall_memory(memory: Memory, key: str):
    value = memory.recall(key)
    if value is None:
        return f"No memory found for '{key}'"

    return f"The value of {key} = {value}"

In [209]:
recall_tool = Tool(
    recall_memory,
    "Recalls the memory from the agent's memory"
)

In [210]:
tool_list = [
    calculator_tool,
    time_tool,
    greet_tool,
    memory_tool,
    recall_tool
]

In [211]:
agent = Agent(
    client=client,
    tool_list=tool_list,
    model = MODEL,
    system_prompt = """You are a helpful AI agent.

You have access to tools for calculations, getting the current time,
and managing memory.

Use the calculator for mathematical calculations.
Use the time tool when the user asks for the current time.

Memory rules:
- If the user explicitly asks you to remember something, use save_memory.
- If the user asks about something that may be stored in memory, use recall_memory.
- Do not invent memories.
- If a requested memory does not exist, clearly tell the user."""
)

In [212]:
while True:
    try:
        user_input = input("You: ")
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input.strip():
        continue

    if user_input.lower().strip() == "exit":
        break

    try:
        response = agent.run(user_input)
        print(response)
    except Exception as e:
        print("Error:", e)


Tool is running
('AI: ', "I've remembered that your native language is Telugu. I'll keep this in mind for our future conversations!")
Tool is running
('AI: ', "I don't have your name stored in my memory yet. Would you like to tell me your name so I can remember it?")
Tool is running
('AI: ', 'Your native language is **Telugu**.')
